# Урок 8.1. Финальный проект: архитектура и требования

Модуль 8 — финал курса «LLM для Python-разработчиков». Семь модулей мы собирали
арсенал; теперь — один настоящий проект, в котором всё это встретится:
**Docs Assistant** — AI-ассистент по внутренней документации компании.

Модуль устроен иначе, чем предыдущие: рядом лежит папка **`project/`** —
полностью рабочее FastAPI-приложение (запускается, тестируется, отвечает).
Шесть уроков модуля — это разбор проекта слой за слоем: не «напишем когда-нибудь»,
а «вот код, вот почему он такой». В этом уроке — ТЗ, архитектура и решения.

🎯 **Цели урока:**

- Прочитать и понять ТЗ: что сервис обязан уметь, а что — осознанно нет.
- Увидеть архитектуру на трёх уровнях (C4: контекст → контейнеры → компоненты).
- Разобрать выбор стека и формат ADR — как фиксировать решения, чтобы через
  полгода не спорить заново.
- Понять критерии приёмки — как узнаем, что проект готов.

## 1. Задача и ТЗ

Легенда прежняя — «Векторика». У компании накопилась внутренняя инженерная
документация: деплой, инциденты, миграции, секреты, код-ревью... Новички задают
в чатах вопросы, ответы на которые давно записаны; старожилы ищут по 12 файлам
глазами. Нужен ассистент.

**Техническое задание.**

Функциональные требования:

1. **Поиск** `POST /api/search`: по вопросу на русском языке вернуть top-k
   релевантных фрагментов документации с указанием источника и оценкой.
2. **Ответ** `POST /api/generate`: по вопросу сгенерировать связный ответ
   **только по документации**, с нумерованными ссылками на источники `[1]`;
   если ответа в документации нет — честно отказаться.
3. **Стриминг**: тот же `/api/generate` со `stream=true` отдаёт ответ потоком
   (SSE), причём источники приходят *до* первого токена.
4. **История** `GET /api/history`: последние пары вопрос-ответ с источниками —
   сырьё для будущего golden-набора (урок 3.3 научил: без истории нет eval).

Нефункциональные:

- **Приватность**: документация не покидает контур — только локальные модели.
- **Бюджет**: ноль. Ollama + qwen2.5:3b, e5-эмбеддинги, Qdrant.
- **Эксплуатируемость**: `/health` и `/ready`, Prometheus-метрики, сквозной
  request-id, конфигурация через окружение — всё, чему научил модуль 5.
- **Тестируемость**: тесты гоняются без GPU, без моделей и без запущенных
  Qdrant/Ollama — быстро и в любом CI.

Осознанно НЕ входит в v1 (и зафиксировано в уроке 8.6 «Компромиссы»):
авторизация и мультипользовательность, автопереиндексация по расписанию,
кэширование ответов, свой UI.

## 2. Архитектура: три уровня C4

C4 — способ рисовать архитектуру уровнями зума. Нам хватит трёх.

**Уровень 1. Контекст** — кто с кем разговаривает:

```
 инженер «Векторики» ──HTTP──▶ [ Docs Assistant ] ──▶ Qdrant   (поиск по векторам)
      (curl / скрипт / IDE)          │
                                     ├──▶ Ollama   (локальная LLM qwen2.5:3b)
                                     └──▶ SQLite   (история диалогов)
```

**Уровень 2. Контейнеры** — процессы и хранилища:

```
 docker compose / локально:
 ┌─────────────────────┐   ┌─────────────┐   ┌──────────────┐
 │ app: FastAPI:8000   │──▶│ qdrant:6333 │   │ ollama:11434 │
 │  + модели e5 и CE   │   │  (HNSW)     │   │  (qwen 3B)   │
 │  внутри процесса    │   └─────────────┘   └──────────────┘
 └─────────┬───────────┘
           └──▶ data/history.db (файл SQLite)
```

Заметьте: эмбеддер и reranker живут **внутри процесса приложения** (это Python-
библиотеки), а LLM и векторная БД — **отдельные серверы**. Разница важна для
health-check'ов (урок 8.5) и масштабирования (урок 8.6).

**Уровень 3. Компоненты** — слои внутри приложения:

```
 routers/            search.py   generate.py   health.py      ← HTTP-граница,
    │                    └────────────┬────────────┘             Pydantic-контракты
 services/                      RagService                     ← оркестрация пайплайна
    │                ┌───────┬───────┴───────┬──────────┐
             EmbeddingService  VectorStore  RerankerService  OllamaClient
                     │              │                            │
              sentence-transf.   Qdrant                     HTTP /api/chat
                                                    HistoryRepository ─▶ SQLite
```

Правило зависимости — только сверху вниз: роутеры знают про `RagService`,
но не про Qdrant; сервисы не знают про HTTP. Это то, что позволит в уроке 8.5
тестировать HTTP-слой с фейковым `RagService`, а `RagService` — с фейковым LLM.

## 3. Выбор стека и ADR

Каждое крупное решение в проекте записано как **ADR** (Architecture Decision
Record) в [`project/docs/adr/`](project/docs/adr/). Формат — полстраницы:
контекст → решение → альтернативы → последствия. Смысл ADR не в бюрократии,
а в том, что через полгода на вопрос «а почему тут SQLite?!» отвечает документ,
а не археология в git blame.

| ADR | Решение | Ключевой довод |
|---|---|---|
| [0001](project/docs/adr/0001-stack.md) | FastAPI + Qdrant + Ollama | приватность (модуль 5: свои модели) + нулевой бюджет |
| [0002](project/docs/adr/0002-embeddings.md) | e5-small, 384 dim, CPU | русский язык, ~120 МБ, изучена в модуле 2 |
| [0003](project/docs/adr/0003-reranker.md) | cross-encoder, выключаемый | двухступенчатый поиск из урока 2.6/4.4; чест­ные A/B |
| [0004](project/docs/adr/0004-history-sqlite.md) | история в SQLite | ноль эксплуатации; узкий async-интерфейс для миграции на PG |

Обратите внимание на скелет каждого ADR: раздел «Альтернативы» обязателен.
ADR без отвергнутых альтернатив — это не решение, а пресс-релиз.

## 4. Знания проекта: 12 документов

База знаний — [`project/data/docs/`](project/data/docs/): 12 markdown-документов
внутренней документации «Векторики». Посмотрим, с чем работаем.

In [ ]:
from pathlib import Path

DOCS = Path("project/data/docs")
total = 0
for path in sorted(DOCS.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ")
    sections = sum(1 for line in text.splitlines() if line.startswith("## "))
    total += len(text)
    print(f"{path.name:<28} {len(text):>5} симв., {sections} секций | {title}")
print(f"\nвсего: {total} символов ≈ {total // 4} токенов")

Документы намеренно **структурированные** (заголовок `#`, секции `##`) и
**фактурные**: конкретные времена реакции, номера портов, названия job'ов.
Ассистент по водянистой документации не может быть лучше неё самой — мусор на
входе, мусор на выходе, и никакой RAG это не чинит (урок 4.1, антипаттерны).

И масштаб честности: вся база — ~16 тысяч символов, около 60 чанков. Это
**демо-масштаб**: решения проекта (Qdrant, ANN, reranker) выбраны так, чтобы
пережить рост до тысяч документов, но мерить «скорость поиска» на 60 чанках —
самообман (урок 2.2 объяснял, когда ANN начинает окупаться).

## 5. Критерии приёмки

Как узнаем, что проект готов? Чек-лист, который закроем к концу модуля:

1. `pytest` зелёный **без** Qdrant, Ollama и скачанных моделей (урок 8.5).
2. `scripts/index_docs.py` идемпотентен: повторный запуск не плодит дублей (8.2).
3. `/api/search` возвращает релевантные фрагменты; reranker включается
   и выключается конфигом и параметром запроса (8.3).
4. `/api/generate` отвечает только по документации, со ссылками `[N]`,
   и честно отказывается на вопрос не из базы (8.4).
5. Стриминг отдаёт источники до первого токена; история пишется даже при
   обрыве стрима (8.4).
6. `/ready` честно деградирует при выключенной зависимости (8.5).
7. Все «почему» записаны: ADR + урок 8.6 про компромиссы.

## Мини-квиз

**1. Почему эмбеддер — внутри процесса приложения, а LLM — отдельный сервер?**

<details><summary>Ответ</summary>

Эмбеддер маленький (~120 МБ) и нужен на каждый запрос с минимальной задержкой —
дешевле держать в процессе. LLM тяжёлая, требует GPU/много RAM и уже
эксплуатируется как отдельный сервис (Ollama) с собственной очередью и
жизненным циклом (модуль 5). Граница процесса = граница масштабирования.
</details>

**2. Зачем в ADR раздел «Альтернативы», если решение уже принято?**

<details><summary>Ответ</summary>

Чтобы решение можно было пересмотреть осмысленно: когда контекст изменится
(вырос трафик, появился GPU-сервер), по ADR видно, какие варианты отвергли
и почему — и не изменились ли эти «почему». Без альтернатив ADR не защищает
от повторного спора.
</details>

**3. История диалогов — функциональное требование или задел? Зачем она в v1?**

<details><summary>Ответ</summary>

Задел под качество: из истории вырастет golden-набор для eval (урок 3.3) и
разбор плохих ответов. Убрать её из v1 легко, а восстановить задним числом
вопросы пользователей — невозможно. Дёшево сейчас, бесценно потом.
</details>

## 🎓 Итоги

- ТЗ: поиск + генерация с цитированием + стриминг + история; приватно, бесплатно,
  эксплуатируемо, тестируемо. Не менее важно «не входит в v1» — записанные
  границы экономят споры.
- C4: контекст (кто с кем) → контейнеры (процессы) → компоненты (слои).
  Зависимости только сверху вниз — это фундамент тестируемости.
- ADR — полстраницы на решение, с обязательными альтернативами. Четыре ADR
  проекта покрывают стек, эмбеддинги, reranker и историю.
- База знаний — 12 структурированных документов, ~60 чанков: демо-масштаб
  с прод-архитектурой.

**Дальше — урок 8.2:** окружение и компоненты — конфигурация через
pydantic-settings, чанкинг markdown и живая индексация в Qdrant.